## Setup

Keep [`example_data.py`](example_data.py), [`data_fidelity_metrics.py`](data_fidelity_metrics.py), and [`missingness_fidelity_metrics.py`](missingness_fidelity_metrics.py) beside this notebook.

In [1]:
#| label: setup
from IPython.display import display

from example_data import make_example_views
from missingness_fidelity_metrics import (
    marginal_missingness_details,
    missingness_hellinger_distance,
)

## 1. Metric and inputs

| Property | Value |
|---|---|
| Metric | Marginal missingness Hellinger distance |
| Family | Missingness fidelity |
| Origin | Devised in the source study |
| Classifier | None |
| Input | Original $X,Y$ through $M_X,M_Y$ |
| Holdout | None |
| Range | $[0,1]$; smaller is better |

This metric compares **how often** each column is missing. It does not compare which columns are missing together in the same row.

## 2. Mathematics

For column $j$, the marginal missingness rates are

$$
r_{Xj}=\frac1{n_X}\sum_{i=1}^{n_X}(M_X)_{ij},
\qquad
r_{Yj}=\frac1{n_Y}\sum_{i=1}^{n_Y}(M_Y)_{ij}.
$$

Each rate defines a Bernoulli distribution over **observed** and **missing**:

$$
p_j=(1-r_{Xj},r_{Xj}),\qquad q_j=(1-r_{Yj},r_{Yj}).
$$

The column distance is

$$
H_j^{\mathrm{marg}}=\sqrt{\frac12\left[
(\sqrt{1-r_{Xj}}-\sqrt{1-r_{Yj}})^2+
(\sqrt{r_{Xj}}-\sqrt{r_{Yj}})^2
\right]}.
$$

Let $A$ contain only columns missing in at least one table. The reported metric is

$$
D_{\mathrm{miss,marg}}(X,Y)=\frac1{|A|}\sum_{j\in A}H_j^{\mathrm{marg}}.
$$

Fully observed columns are excluded so adding unrelated complete columns cannot dilute the result. If $A$ is empty, the distance is defined as zero.

In [2]:
#| label: marginal-missingness

example = make_example_views()
X, Y = example.X, example.Y
M_X, M_Y = example.M_X, example.M_Y

details = marginal_missingness_details(X, Y)
D_marginal = missingness_hellinger_distance(X, Y, mode="marginal")

print("M_X")
display(M_X)
print("M_Y")
display(M_Y)
display(details.round(4))
print(f"Marginal missingness Hellinger distance = {D_marginal:.6f}")
assert abs(D_marginal - details["distance"].mean()) < 1e-12

M_X


,age,income_k,region,outcome
x1,0,0,0,0
x2,0,0,0,0
x3,1,0,0,0
x4,0,1,0,0
x5,0,0,1,0
x6,0,0,0,0


M_Y


,age,income_k,region,outcome
y1,0,0,0,0
y2,1,0,0,0
y3,0,1,0,0
y4,0,0,1,0
y5,0,0,0,0


,column,P_X(missing),P_Y(missing),distance
0,age,0.1667,0.2,0.0305
1,income_k,0.1667,0.2,0.0305
2,region,0.1667,0.2,0.0305


Marginal missingness Hellinger distance = 0.030483


## 3. Interpretation

The shared example gives $D_{\mathrm{miss,marg}}(X,Y)\approx0.0305$: the three incomplete columns have similar, but not identical, missing rates. The metric is symmetric and supports unequal row counts. A zero value guarantees matching per-column missing rates, but not matching row-wise patterns.

## Takeaway

Convert each incomplete column into a Bernoulli observed/missing distribution, compute its Hellinger distance, and average only over columns with missingness.